In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import glob
import shutil

# 1. Define your base paths
# Update this if your main folder is named differently in Drive
base_dir = '/content/drive/MyDrive/CPTAC-LSCC_split_balanced'
unused_normal = os.path.join(base_dir, 'unused', 'NORMAL')
unused_tumor = os.path.join(base_dir, 'unused', 'TUMOR')

# Ensure the unused directories exist
os.makedirs(unused_normal, exist_ok=True)
os.makedirs(unused_tumor, exist_ok=True)

# 2. Define the new balanced targets
target_counts = {'train': 266, 'val': 57, 'test': 57}
removed_files = {'NORMAL': [], 'TUMOR': []}

print("🚀 Starting dataset re-balancing...\n")

# 3. Explicitly surgically remove the empty validation slide first
bad_slide_path = os.path.join(base_dir, 'val', 'NORMAL', 'C3N-03762-28.svs')
if os.path.exists(bad_slide_path):
    shutil.move(bad_slide_path, os.path.join(unused_normal, 'C3N-03762-28.svs'))
    removed_files['NORMAL'].append('C3N-03762-28.svs (Explicitly requested empty slide)')
else:
    print(f"⚠️ {bad_slide_path} not found. (It may have already been moved).\n")

# 4. General function to trim the extra slides from the END of the folders
def trim_folder(split_name, category, target_count):
    folder_path = os.path.join(base_dir, split_name, category)
    unused_path = unused_normal if category == 'NORMAL' else unused_tumor

    # Get sorted list of files so we can reliably pick from the end
    files = sorted(glob.glob(os.path.join(folder_path, '*.svs')))
    current_count = len(files)

    if current_count > target_count:
        num_to_remove = current_count - target_count

        # Slice from the end of the list
        files_to_remove = files[-num_to_remove:]

        for f in files_to_remove:
            filename = os.path.basename(f)
            shutil.move(f, os.path.join(unused_path, filename))
            removed_files[category].append(f"{filename} (from {split_name})")

# 5. Run the trimming logic across all folders
for split in ['train', 'val', 'test']:
    trim_folder(split, 'NORMAL', target_counts[split])
    trim_folder(split, 'TUMOR', target_counts[split])

# 6. Print the definitive ledger of what was removed
print("🎉 Re-balancing Complete! Here is the ledger of moved files:\n")

print(f"🔴 Removed TUMOR slides ({len(removed_files['TUMOR'])} total):")
for f in removed_files['TUMOR']:
    print(f"  - {f}")

print(f"\n🟢 Removed NORMAL slides ({len(removed_files['NORMAL'])} total):")
for f in removed_files['NORMAL']:
    print(f"  - {f}")

🚀 Starting dataset re-balancing...

🎉 Re-balancing Complete! Here is the ledger of moved files:

🔴 Removed TUMOR slides (7 total):
  - C3N-03923-22.svs (from train)
  - C3N-04155-22.svs (from train)
  - C3N-04155-23.svs (from train)
  - C3N-04167-22.svs (from train)
  - C3N-04609-22.svs (from train)
  - C3N-05079-21.svs (from val)
  - C3N-05915-21.svs (from test)

🟢 Removed NORMAL slides (7 total):
  - C3N-03762-28.svs (Explicitly requested empty slide)
  - C3N-04170-26.svs (from train)
  - C3N-04179-29.svs (from train)
  - C3N-04609-26.svs (from train)
  - C3N-04701-29.svs (from train)
  - C3N-05629-26.svs (from train)
  - C3N-05929-25.svs (from test)


In [ ]:
import os
from pathlib import Path

# Define the exact Google Drive paths
data_dir = Path('/content/drive/MyDrive/CPTAC-LSCC_split_balanced')

splits = {
    'Train': [data_dir / 'train' / 'TUMOR', data_dir / 'train' / 'NORMAL'],
    'Validation': [data_dir / 'val' / 'TUMOR', data_dir / 'val' / 'NORMAL'],
    'Test': [data_dir / 'test' / 'TUMOR', data_dir / 'test' / 'NORMAL']
}

# Dictionary to hold unique sets of Patient IDs for each split
patient_ids = {'Train': set(), 'Validation': set(), 'Test': set()}
slide_counts = {'Train': 0, 'Validation': 0, 'Test': 0}

def extract_patient_id(filename):
    """
    Extracts the patient ID from a CPTAC file.
    Example: C3N-05915-21.svs -> C3N-05915
             C3L-12345-01.svs -> C3L-12345
    """
    # Check if the file matches the expected CPTAC prefix pattern
    if filename.startswith('C3N') or filename.startswith('C3L'):
        parts = filename.split('-')
        # Ensure there are at least two parts to form the patient ID
        if len(parts) >= 2:
            return f"{parts[0]}-{parts[1]}"
    return None

# Scan directories and populate the patient ID sets
for split_name, paths in splits.items():
    for folder_path in paths:
        if folder_path.exists():
            for file_path in folder_path.iterdir():
                if file_path.is_file():
                    slide_counts[split_name] += 1

                    patient_id = extract_patient_id(file_path.name)
                    if patient_id:
                        patient_ids[split_name].add(patient_id)
        else:
            print(f"Warning: Folder not found -> {folder_path}")

# Calculate intersections to find overlaps (data leaks)
train_val_leak = patient_ids['Train'].intersection(patient_ids['Validation'])
train_test_leak = patient_ids['Train'].intersection(patient_ids['Test'])
val_test_leak = patient_ids['Validation'].intersection(patient_ids['Test'])

# Output the results
print("=== CPTAC Patient-Level Leakage Report ===\n")
print(f"Total Unique Patients in Train: {len(patient_ids['Train'])}")
print(f"Total Unique Patients in Validation: {len(patient_ids['Validation'])}")
print(f"Total Unique Patients in Test: {len(patient_ids['Test'])}\n")

if train_val_leak:
    print(f"⚠️ LEAK DETECTED: {len(train_val_leak)} patients overlap between Train and Validation!")
    print(f"Overlapping IDs: {train_val_leak}\n")
else:
    print("✅ No patient overlap between Train and Validation.")

if train_test_leak:
    print(f"⚠️ LEAK DETECTED: {len(train_test_leak)} patients overlap between Train and Test!")
    print(f"Overlapping IDs: {train_test_leak}\n")
else:
    print("✅ No patient overlap between Train and Test.")

if val_test_leak:
    print(f"⚠️ LEAK DETECTED: {len(val_test_leak)} patients overlap between Validation and Test!")
    print(f"Overlapping IDs: {val_test_leak}\n")
else:
    print("✅ No patient overlap between Validation and Test.")


print("\n--- Data Breakdown ---\n")
print(f"Train      -> Slides: {slide_counts['Train']:<5} | Unique Patients: {len(patient_ids['Train'])}")
print(f"Validation -> Slides: {slide_counts['Validation']:<5} | Unique Patients: {len(patient_ids['Validation'])}")
print(f"Test       -> Slides: {slide_counts['Test']:<5} | Unique Patients: {len(patient_ids['Test'])}\n")

=== TCGA Patient-Level Leakage Report ===

Total Unique Patients in Train: 138
Total Unique Patients in Validation: 31
Total Unique Patients in Test: 35

✅ No patient overlap between Train and Validation.
✅ No patient overlap between Train and Test.
✅ No patient overlap between Validation and Test.

--- Data Breakdown ---

Train      -> Slides: 532   | Unique Patients: 138
Validation -> Slides: 114   | Unique Patients: 31
Test       -> Slides: 114   | Unique Patients: 35

